# Week 2: Exploratory Data Analysis (EDA) — Retail Store Sales
**Prepared by:** Tushar Sharma  
**Tools:** Python, pandas, NumPy, Matplotlib, SciPy  
**Dataset:** Retail Store Sales dataset used in Week 1 data cleaning.

## Objective
Explore transaction-level retail data using descriptive statistics, visualizations, time-based summaries, and an initial hypothesis test. This notebook is designed to be reproducible. It reports missingness explicitly and does not silently replace unknown values.

## Dataset note
This analysis uses the cleaned output from Week 1 (`retail_store_sales_cleaned.csv`). Place this notebook and CSV in the same project folder, or update the file path in the next cell. Missing `Item`, `Quantity`, `Total Spent`, and `Discount Applied` values are retained where they cannot be safely inferred. Recorded spend totals therefore describe available `Total Spent` values, not necessarily every transaction's complete spend.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Locate the cleaned dataset relative to this notebook
data_path = Path("retail_store_sales_cleaned.csv")
if not data_path.exists():
    data_path = Path("../output/retail_store_sales_cleaned.csv")

df = pd.read_csv(data_path)
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
df["Discount Applied"] = df["Discount Applied"].astype("boolean")

for col in ["Price Per Unit", "Quantity", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

## 1. Dataset overview
Check the dimensions, column types, date range, and missing values before interpreting any patterns.

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn data types:")
print(df.dtypes)
print("\nDate range:", df["Transaction Date"].min(), "to", df["Transaction Date"].max())
print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))
print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate transaction IDs:", df["Transaction ID"].duplicated().sum())

**Interpretation:** The dataset has 12,575 rows and 11 columns. The main remaining gaps are `Discount Applied`, `Item`, `Quantity`, and `Total Spent`. Missing values are not automatically errors: some cannot be inferred from the other columns. Exact duplicate rows and duplicate transaction IDs should be checked separately.

## 2. Descriptive statistics
Use count, mean, standard deviation, quartiles, minimum, and maximum to summarize numerical variables.

In [ ]:
numeric_cols = ["Price Per Unit", "Quantity", "Total Spent"]
display(df[numeric_cols].describe().T)

print("\nMedian values:")
print(df[numeric_cols].median())

**Interpretation:** `Total Spent` has a mean of about 129.65 and a median of 108.50 among recorded values. The mean being higher than the median suggests that larger transactions pull the average upward. The standard deviation is substantial, showing transaction spend varies considerably.

## 3. Univariate analysis: distribution of transaction spend

In [ ]:
plt.figure(figsize=(8, 4.5))
df["Total Spent"].dropna().hist(bins=30)
plt.title("Distribution of Transaction Spend")
plt.xlabel("Total Spent")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.show()

**Inference:** The histogram helps show where transaction values cluster and whether the distribution is symmetric or has a long tail. Because the mean exceeds the median, the spend distribution is likely right-skewed. Missing `Total Spent` values are excluded from this plot.

## 4. Category-level comparison

In [ ]:
category_summary = (
    df.groupby("Category")["Total Spent"]
      .agg(known_transactions="count", total_recorded_spend="sum", average_spend="mean")
      .sort_values("total_recorded_spend", ascending=False)
)
display(category_summary.round(2))

category_summary["total_recorded_spend"].sort_values().plot(kind="barh", figsize=(9, 5))
plt.title("Total Recorded Spend by Product Category")
plt.xlabel("Total Spent")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

**Inference:** Butchers has the highest recorded total spend in this dataset, followed by Electric household essentials. Milk Products has the lowest recorded total spend. This is descriptive, not proof that one category is inherently more profitable: prices, missing totals, and transaction mix can affect the totals.

## 5. Quantity and spend relationship

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(df["Quantity"], df["Total Spent"], alpha=0.25)
plt.title("Quantity vs Total Spent")
plt.xlabel("Quantity")
plt.ylabel("Total Spent")
plt.tight_layout()
plt.show()

print("Correlation matrix:")
display(df[["Price Per Unit", "Quantity", "Total Spent"]].corr().round(3))

**Inference:** Quantity and Total Spent should generally move together because total spend is calculated from unit price multiplied by quantity. This relationship is partly mathematical by construction, so correlation should not be presented as a surprising causal discovery.

## 6. Correlation heat map

In [ ]:
corr = df[["Price Per Unit", "Quantity", "Total Spent"]].corr()
plt.figure(figsize=(6, 4.5))
plt.imshow(corr, vmin=-1, vmax=1, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=25, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.colorbar(label="Correlation")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 7. Discount comparison and hypothesis test
Compare recorded transaction spend for rows marked `True` versus `False`. Rows with unknown discount status are excluded. Welch's two-sample t-test does not assume equal group variances.

In [ ]:
discount_df = df.dropna(subset=["Discount Applied", "Total Spent"])
with_discount = discount_df.loc[discount_df["Discount Applied"] == True, "Total Spent"]
without_discount = discount_df.loc[discount_df["Discount Applied"] == False, "Total Spent"]

print("With discount: n =", len(with_discount),
      "| mean =", round(with_discount.mean(), 2),
      "| median =", round(with_discount.median(), 2))
print("Without discount: n =", len(without_discount),
      "| mean =", round(without_discount.mean(), 2),
      "| median =", round(without_discount.median(), 2))

test = stats.ttest_ind(with_discount, without_discount, equal_var=False, nan_policy="omit")
print("Welch t-statistic:", round(test.statistic, 4))
print("p-value:", round(test.pvalue, 4))

plt.figure(figsize=(7, 4.5))
plt.boxplot([without_discount, with_discount],
            labels=["No discount", "Discount applied"], showfliers=False)
plt.title("Transaction Spend by Discount Status")
plt.ylabel("Total Spent")
plt.tight_layout()
plt.show()

### Hypotheses
- **H₀ (null):** The mean recorded transaction spend is equal for the discount and no-discount groups.
- **H₁ (alternative):** The mean recorded transaction spend differs between the groups.
- **Decision rule:** Use α = 0.05.

**Result on this dataset:** The group means are very close (about 130.49 with discount versus 129.95 without discount), and the p-value is about 0.799. Since p > 0.05, fail to reject H₀. This sample does not provide statistically significant evidence of a difference in mean transaction spend between the two recorded groups. This does **not** prove the groups are identical, and it does not establish a causal effect of discounts. About one-third of discount values are missing, so selection bias is possible.

## 8. Location and payment method summaries

In [ ]:
location_summary = df.groupby("Location")["Total Spent"].agg(
    known_transactions="count", total_recorded_spend="sum", average_spend="mean"
).sort_values("total_recorded_spend", ascending=False)
display(location_summary.round(2))

payment_summary = df["Payment Method"].value_counts(dropna=False).to_frame("transactions")
payment_summary["percentage"] = payment_summary["transactions"] / len(df) * 100
display(payment_summary.round(2))

**Inference:** Online has a slightly higher recorded total spend than In-store, but the difference is small and the number of known-spend transactions differs. Cash is the most frequently recorded payment method, followed by Digital Wallet and Credit Card. Counts describe transaction frequency, not customer preference at population level.

## 9. Time-based analysis

In [ ]:
dated = df.dropna(subset=["Transaction Date", "Total Spent"]).copy()
dated["Month"] = dated["Transaction Date"].dt.to_period("M").astype(str)
monthly_spend = dated.groupby("Month")["Total Spent"].sum().sort_index()

monthly_spend.plot(figsize=(9, 4.5))
plt.title("Monthly Recorded Spend Over Time")
plt.xlabel("Month")
plt.ylabel("Total Spent")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Recorded spend by year:")
display(dated.assign(Year=dated["Transaction Date"].dt.year)
        .groupby("Year")["Total Spent"].agg(["count", "sum", "mean"]).round(2))

**Caution:** The available dates span 2022-01-01 through 2025-01-18. The 2025 period is partial, so its total must not be compared directly with complete years without accounting for the shorter period. Month-to-month totals can also be affected by missing spend values.

## 10. Key findings, limitations, and conclusion
1. The data contains 12,575 transactions and 11 fields; there are no exact duplicate rows or duplicate transaction IDs in the cleaned dataset.
2. The recorded average spend is approximately 129.65, with a median of 108.50, suggesting larger transactions raise the mean.
3. Butchers has the highest recorded category spend; Milk Products has the lowest.
4. Online has slightly higher recorded spend than In-store; Cash is the most frequent payment method.
5. Welch's t-test did not find a statistically significant difference in mean spend by recorded discount status (p ≈ 0.799, α = 0.05).
6. Missing `Item`, `Quantity`, `Total Spent`, and `Discount Applied` values limit some comparisons. Missing values were not filled with unsupported guesses.
7. The dataset appears synthetic/structured, and findings should not be generalized to a real retail business without validating the data source.

**Conclusion:** This EDA combines descriptive statistics, visual analysis, time summaries, and a hypothesis test. It provides a reproducible starting point for further feature engineering or predictive modeling, while making data-quality limitations explicit.